# Open models on a GPU notebook

Generates the replies of the two open models for `runs/run.py --import` (see `runs/README.md`).

1. GPU: at least 2 × 16 GB, or one GPU with 24 GB or more (Kaggle: GPU T4 x2 with Internet on; Colab: L4 or A100).
   One 16 GB T4 is not enough for Qwen2.5-Coder-7B in 16-bit precision.
2. Upload `generate_open.py` and the requests file `requests_<RUN>.jsonl` that `runs/run.py MODEL --export` wrote
   (on Kaggle as a dataset; on Colab into the working folder), and set `RUN` below.
3. Run all cells. If the session is cut off, run them again: lines already written are skipped.
4. Download `<RUN>_outputs.tar.gz`, which holds the two `replies_*_<RUN>.jsonl` files and their `.meta.json`
   files.


In [ ]:
import glob

RUN = "probe"  # the requests file is requests_<RUN>.jsonl

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name))
    if not hits:
        raise FileNotFoundError(f"upload {name} first")
    return hits[0]

SCRIPT = find("generate_open.py")
REQUESTS = find(f"requests_{RUN}.jsonl")
print(SCRIPT, REQUESTS, sep="\n")


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
try:
    import accelerate
except ImportError:
    !pip install -q accelerate
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__, "GPUs", torch.cuda.device_count())


In [ ]:
!python {SCRIPT} Qwen/Qwen2.5-Coder-7B-Instruct {REQUESTS} replies_qwen2.5-coder-7b_{RUN}.jsonl --batch 8


In [ ]:
!python {SCRIPT} codellama/CodeLlama-7b-Instruct-hf {REQUESTS} replies_codellama-7b_{RUN}.jsonl --batch 2


In [ ]:
!ls -la replies_*_{RUN}.jsonl*
!tar czf {RUN}_outputs.tar.gz replies_*_{RUN}.jsonl*
